# CpDP experiments

Runs the sweeps, aggregates one-dimensional statistics, and renders figures.
Everything here reads from `results/sweep.csv` and `output/`, so re-running the
analysis never re-solves anything.

In [ ]:
import os, sys, glob, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "src")
from tsgo.experiments.grids import STANDARD, TETRAHEDRON, grid_for

SWEEP = "results/sweep.csv"
pd.set_option("display.width", 160, "display.max_columns", 40)
print("standard grid   :", STANDARD)
print("tetrahedron grid:", TETRAHEDRON)

## 1. Run the sweep

Interruptible: rerunning skips cells already in the CSV. Start with one
container to gauge timing before launching the full set overnight.

In [ ]:
# One container first -- adjust and re-run, or use ./run_experiments.sh for everything.
!tsgo-batch --containers tetrahedron --p 1-6 --out {SWEEP} --packing-density 0.25

In [ ]:
# The whole sweep (long). Uncomment to run.
# !./run_experiments.sh

## 2. One-dimensional statistics

`D_best` is the headline number -- packing results report best-ever-found, and a
mean would understate the method against analytic optima. Mean, std and success
rate describe reliability; the time columns describe cost.

In [ ]:
df = pd.read_csv(SWEEP)
df["feasible"] = df["feasible"].astype(str).str.lower().eq("true")
print(f"{len(df)} runs, {df.feasible.mean():.1%} feasible")

ok = df[df.feasible]
stats = (ok.groupby(["container", "metric", "p"])
           .agg(D_best=("D", "max"), D_mean=("D", "mean"), D_std=("D", "std"),
                D_min=("D", "min"),
                t_mean=("wall_time_s", "mean"), t_median=("wall_time_s", "median"),
                t_std=("wall_time_s", "std"), t_max=("wall_time_s", "max"),
                n=("D", "size"))
           .reset_index())
stats["success_rate"] = (df.groupby(["container","metric","p"]).feasible.mean()
                           .reindex(pd.MultiIndex.from_frame(stats[["container","metric","p"]])).values)
stats["D_cv"] = stats.D_std / stats.D_mean
stats.round(8)

### Monotonicity check

D(p) must be non-increasing: any feasible (p+1)-configuration restricts to a
feasible p-configuration with the same or larger D. A violation means the
smaller p is under-optimised.

In [ ]:
for (c, m), g in stats.sort_values("p").groupby(["container", "metric"]):
    d = g.D_best.values; ps = g.p.values
    bad = [(ps[i], d[i], ps[i+1], d[i+1]) for i in range(len(d)-1) if d[i+1] > d[i] + 1e-8]
    print(f"{c:<13}{m:<10}" + ("OK" if not bad else f"{len(bad)} violation(s): " +
          ", ".join(f"D({a})={b:.8f} < D({c2})={d2:.8f}" for a,b,c2,d2 in bad)))

## 3. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for c, g in stats[stats.metric == "EUCLIDEAN"].sort_values("p").groupby("container"):
    axes[0].plot(g.p, g.D_best, "o-", label=c, ms=4)
    axes[0].fill_between(g.p, g.D_min, g.D_best, alpha=0.15)
    axes[1].plot(g.p, g.t_median, "o-", label=c, ms=4)
axes[0].set(xlabel="p", ylabel="D (radius)", title="Best D vs p, band = min-max over seeds")
axes[1].set(xlabel="p", ylabel="median wall time (s)", yscale="log", title="Runtime scaling")
for a in axes: a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Tetrahedral numbers

A regular tetrahedron packs 1, 4, 10, 20, 35, 56, 84 points in symmetric
layered configurations, so D(p) should step visibly at those counts.

In [ ]:
from tsgo.experiments.grids import tetrahedral_numbers
t = stats[(stats.container == "tetrahedron") & (stats.metric == "EUCLIDEAN")].sort_values("p")
if len(t):
    plt.figure(figsize=(7, 4.5))
    plt.plot(t.p, t.D_best, "o-", ms=4, label="D best")
    for n in tetrahedral_numbers(int(t.p.max())):
        plt.axvline(n, color="crimson", ls=":", alpha=.6)
    plt.xlabel("p"); plt.ylabel("D"); plt.title("Tetrahedron; dotted lines = tetrahedral numbers")
    plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

## 4. Figures for the paper

`--clean` strips axes, ticks, gridlines and panes. Camera angles are explicit,
so figures regenerate identically.

In [ ]:
os.makedirs("figures", exist_ok=True)
for src in sorted(glob.glob("output/*/p*/euclidean/seed0.pkl"))[:6]:
    parts = src.split(os.sep)
    dest = f"figures/{parts[1]}_{parts[2]}.png"
    !tsgo-render {src} --clean --elev 22 --azim 135 --out {dest}

## 5. Gurobi certified bounds (convex containers)

CpDP is non-convex even in a convex container: the dispersion constraint
`||c_i - c_j|| >= 2D` is reverse-convex, so there are many local optima and a
local solver certifies nothing. Gurobi's spatial branch-and-bound maintains a
valid upper bound, so a closed gap *proves* optimality. Expect it to become
intractable past p ~ 8-10 -- that is the reportable result.

Needs `pip install -e ".[gurobi]"` and a licence.

In [ ]:
!tsgo-compare --containers cube tetrahedron --p 2-6 --seeds 5 --time-limit 300 --out results/compare.csv

In [ ]:
if os.path.exists("results/compare.csv"):
    cmp = pd.read_csv("results/compare.csv")
    display(cmp[["container","p","D_tsgo_best","D_gurobi","D_gurobi_bound",
                 "rel_err_vs_bound","certified_optimal","t_tsgo_total_s","t_gurobi_s"]].round(8))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for c, g in cmp.groupby("container"):
        ax[0].plot(g.p, g.rel_err_vs_bound, "o-", label=c, ms=4)
        ax[1].plot(g.p, g.t_tsgo_total_s, "o-", label=f"{c} TSGO", ms=4)
        ax[1].plot(g.p, g.t_gurobi_s, "s--", label=f"{c} Gurobi", ms=4)
    ax[0].set(xlabel="p", ylabel="gap to certified bound (%)", title="Distance from optimum")
    ax[1].set(xlabel="p", ylabel="wall time (s)", yscale="log", title="TSGO vs Gurobi")
    for a in ax: a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 6. LaTeX table

In [ ]:
tab = (stats[stats.metric == "EUCLIDEAN"]
       .pivot_table(index="p", columns="container", values="D_best")
       .round(8))
print(tab.to_latex(escape=False, na_rep="--",
                   caption="Best radius $D$ by container and $p$.", label="tab:D_best"))